# Puzzle

https://thefiddler.substack.com/p/can-you-or-cantor-you-find-the-distance

### This Week’s Fiddler

Suppose I independently pick two random points in the Cantor set. On average, how far apart can I expect them to be?

### This Week’s Extra Credit

Suppose I independently pick three random points in the Cantor set. Each point has some value between 0 and 1.

What is the probability that these three values can be the side lengths of a triangle?

# Numerical Solution

Code is simple here, and the extra credit will benefit from some code anyway, so let's just go that route first.

In [ ]:
import math
import random

LEVELS = 40 # number of levels in our cantor tree

twice_powers_of_one_third = [2.0 / math.pow(3, i) for i in range(LEVELS)]

def gen_cantor_number():
    """Generates a random number in the Cantor set."""
    result = 0.0
    for i in range(1, LEVELS):
        # Randomly decide whether to include the left or right segment
        if random.choice([True, False]):
            result += twice_powers_of_one_third[i]
    return result

In [ ]:
# Since each cantor number needs a lot of random calls, we generate a list of them first to speed up the later calculations.
ELEMENTS = 1000000
cantor_numbers = [gen_cantor_number() for _ in range(ELEMENTS)]

PASSES = 10
TRIALS = 1000000
global_sum_of_distances = 0.0
for p in range(PASSES):
    sum_of_distances = 0.0
    for t in range(TRIALS):
        a, b = random.sample(cantor_numbers, 2)
        distance = abs(a - b)
        sum_of_distances += distance
    average_distance = sum_of_distances / TRIALS
    print(f"Average distance between two random numbers in the Cantor set (pass {p}): {average_distance}")
    global_sum_of_distances += average_distance
print(f"Global average distance between two random numbers in the Cantor set: {global_sum_of_distances / PASSES}")

Average distance between two random numbers in the Cantor set (pass 0): 0.39968364656544353
Average distance between two random numbers in the Cantor set (pass 1): 0.39983286682075986
Average distance between two random numbers in the Cantor set (pass 2): 0.3992516302882563
Average distance between two random numbers in the Cantor set (pass 3): 0.3996351962399765
Average distance between two random numbers in the Cantor set (pass 4): 0.39998411774091774
Average distance between two random numbers in the Cantor set (pass 5): 0.40005493739630393
Average distance between two random numbers in the Cantor set (pass 6): 0.4003086039772982
Average distance between two random numbers in the Cantor set (pass 7): 0.3998376697608029
Average distance between two random numbers in the Cantor set (pass 8): 0.39973114222695544
Average distance between two random numbers in the Cantor set (pass 9): 0.4001674106441555
Global average distance between two random numbers in the Cantor set: 0.3998487221660

In [3]:
def satisfies_triangle_inequality(a, b, c):
    """Checks if the triangle inequality holds for three numbers."""
    return (a <= b + c) and (b <= a + c) and (c <= a + b)

TRIALS_TRIANGLE = 1000000
global_satisfactions = 0

for p in range(PASSES):
    satisfactions = 0
    for t in range(TRIALS_TRIANGLE):
        a, b, c = random.sample(cantor_numbers, 3)
        if satisfies_triangle_inequality(a, b, c):
            satisfactions += 1
    print(f"Probability of satisfactions of the triangle inequality: {satisfactions} / {TRIALS_TRIANGLE} = {satisfactions / TRIALS_TRIANGLE}")
    global_satisfactions += satisfactions
print(f"Global probability of satisfactions of the triangle inequality: {global_satisfactions} / {PASSES * TRIALS_TRIANGLE} = {global_satisfactions / (PASSES * TRIALS_TRIANGLE)}")

Probability of satisfactions of the triangle inequality: 401108 / 1000000 = 0.401108
Probability of satisfactions of the triangle inequality: 400898 / 1000000 = 0.400898
Probability of satisfactions of the triangle inequality: 399527 / 1000000 = 0.399527
Probability of satisfactions of the triangle inequality: 400237 / 1000000 = 0.400237
Probability of satisfactions of the triangle inequality: 399573 / 1000000 = 0.399573
Probability of satisfactions of the triangle inequality: 400400 / 1000000 = 0.4004
Probability of satisfactions of the triangle inequality: 399992 / 1000000 = 0.399992
Probability of satisfactions of the triangle inequality: 400429 / 1000000 = 0.400429
Probability of satisfactions of the triangle inequality: 399897 / 1000000 = 0.399897
Probability of satisfactions of the triangle inequality: 399704 / 1000000 = 0.399704
Global probability of satisfactions of the triangle inequality: 4001765 / 10000000 = 0.4001765


Main fiddler answer seems to be 0.4, and the extra credit also seems to be 0.4.

Maybe it is worth figuring out calculate this exactly. :)

# Fiddler Analytical
The first step of the Cantor number construction corresponds to choosing to add either 0 or 2/3. The second step adds either 0 or 2/9, and so on.
This is a natural fit for the base 3 (ternary) number format. Better explained here: https://en.wikipedia.org/wiki/Cantor_set#Construction_and_formula_of_the_ternary_set

>   In arithmetical terms, the Cantor set consists of all real numbers of the unit interval [ 0 , 1 ] that do not require the digit 1 in order to be expressed as a ternary (base 3) fraction.

i.e. Each cantor number can represented as a $0.d_1d_2d_3...$, where each $d_i$ is either 0 or 2. In our construction, each digit is independent and equally likely to be either 0 or 2.

Let the 2 chosen cantor numbers be 

$a = 0.a_1a_2a_3... , a_i \in [0,2]$

$b = 0.b_1b_2b_3... , b_i \in [0,2]$

If we take the difference between them, and don't do any carry / borrow propagation (kinda like https://en.wikipedia.org/wiki/Carry-save_adder ), we'll get :

difference = $c = a - b = 0.c_1c_2c_3 ... , c_i \in [-2,0,2]$ with 0 having probability $1/2$ and 2 and -2 having probability $1/4$.

Notice that the expected value of each digit is 0 because the distribution is symmetric about 0. And correspondingly, the expected (signed) difference is also 0, which is unsurprising.

Now to get the distance, we need to take the absolute value, which means that if the first non-zero digit is negative, we negate all the digits.

distance = $d = abs(c) = 0.d_1d_2d_3... d_i \in [-2,0,2]$, but the first non-zero digit is 0 with probability $1/2$ and 2 with probability $1/2$, and cannot be -2.

To get the expected distance, we note that only the first non-zero digit matters, since all other digits have expected value 0. Since each digit is zero or non-zero with probability $1/2$, we can get that digit $d_i$ are likely to be the first digits with probability $1/2^i$. The i-th leading non-zero digit has a value of $\frac{2}{3^i}$.   The expected distance is the sum of all the cases.

E(distance) 

= $ \sum_{i=1}^{\infty} \frac{1}{2^i} \times \frac{2}{3^i} $

= $ 2 \sum_{i=1}^{\infty} \frac{1}{6^i} $

= $ 2 (1/5) $

= 2/5 = 0.4

# Extra Credit Analytical

?